<a href="https://colab.research.google.com/github/dhruvi013/AI/blob/main/TextRank_for_doc_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
import numpy as np
import networkx as nx
from nltk.tokenize import  word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

In [ ]:
#download necessary nltk resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
text = """I am learning natural language processing. Natural language processing is the important part of the course artificial intelligence
 This domain has seen many recent advances in terms of its execution. It is based on PageRank, which is used by Google to rank web pages in search.
  TextRank builds a graph of sentences, where edges represent similarity.By running the PageRank algorithm on this graph,
  we can extract the most important words for summarization. This technique is widely used in NLP applications."""

In [ ]:
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    sentences = sent_tokenize(text)
    word_frequency=[]

    for sent in sentences:
        words = word_tokenize(sent.lower())
        filtered_words=[word for word in words if word.isalnum() and word not in stop_words]
        word_frequency.append(Counter(filtered_words))

    return sentences, word_frequency

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
preprocess_text(text)

(['I am learning natural language processing.',
  'Natural language processing is the important part of the course artificial intelligence\n This domain has seen many recent advances in terms of its execution.',
  'It is based on PageRank, which is used by Google to rank web pages in search.',
  'TextRank builds a graph of sentences, where edges represent similarity.By running the PageRank algorithm on this graph, \n  we can extract the most important words for summarization.',
  'This technique is widely used in NLP applications.'],
 [Counter({'learning': 1, 'natural': 1, 'language': 1, 'processing': 1}),
  Counter({'natural': 1,
           'language': 1,
           'processing': 1,
           'important': 1,
           'part': 1,
           'course': 1,
           'artificial': 1,
           'intelligence': 1,
           'domain': 1,
           'seen': 1,
           'many': 1,
           'recent': 1,
           'advances': 1,
           'terms': 1,
           'execution': 1}),
  Coun

In [ ]:
def graph_similarity(word_frequency):
    size=len(word_frequency)
    sim_matrix=np.zeros((size,size))


    for i in range(size):
      for j in range(size):
         if i!=j:
            word1= word_frequency[i]
            word2= word_frequency[j]
            common_words=set(word1.keys()).union(set(word2.keys()))

            vect1=np.array([(word1[k]) for k in common_words])
            vect2=np.array([(word2[i]) for i in common_words])

            sim_matrix[i][j]=cosine_similarity([vect1],[vect2])[0,0]
    return sim_matrix

In [ ]:
def textrank_summarization(text,summary_length):
  sentences,word_frequency=preprocess_text(text)
  similarity_matrix=graph_similarity(word_frequency)
  #build the graph
  graph=nx.from_numpy_array(similarity_matrix)
  scores=nx.pagerank(graph)

  ranked_sentences=sorted(((scores[i],s) for i,s in enumerate(sentences)),reverse=True)
  return ranked_sentences[:summary_length]

In [ ]:
textrank_summarization(text,2)

[(0.26886634088940975,
  'Natural language processing is the important part of the course artificial intelligence\n This domain has seen many recent advances in terms of its execution.'),
 (0.22588941482123254, 'I am learning natural language processing.')]

In [ ]:
ranked_sentences=textrank_summarization(text,2)
summary=" ".join([sent for i,sent in ranked_sentences])
print(summary)

Natural language processing is the important part of the course artificial intelligence
 This domain has seen many recent advances in terms of its execution. I am learning natural language processing.
